### Make Gantt chart fr the schedule

Same as version (4) as in the `Gits`/mantid-scripts/` rpository.

Same as version 3 but Summarised in a clean example

Can create the correct Gantt in combining 2 sources of data: The "plan" data as previously and the summary of the ILL database for the number of days, the sample environment, ...

But: A difficulty to match the plan and the ILL database. Few in common, in plan the information is very abreviated. Only the experiment number should match... To extract from the database (easy) and from the plan string (less easy).

A plan file is something like:

```

R	86400 1589328000 0 0 0
N	Fennel-4-01-1654-3d-CO
5/14/2020  0:0:0  0:0:0  0:0:0  0:0:0  ---------- 0 0
R	86400 1589673600 0 0 0
N	OMalley-7-01-505-4d-CF
5/18/2020  0:0:0  0:0:0  0:0:0  0:0:0  ---------- 0 0
R	86400 1589846400 0 0 0
N	Telling-9-13-880-2d-CO
...

From the plan owner web: https://www.bitrot.de/plan.html:

    plan is a schedule planner based on X/Motif. It displays a month calendar similar to xcal, but every day box is large enough to show appointments in small print. By pressing on a day box, the appointments for that day can be listed and edited. Appointments are entered with the following information (everything except the time is optional):

    an optional script to be executed,
    early-warn and late-warn triggers that precede the alarm time
    repetitions: [n-th] weekdays, days-of-the-month, every n days, yearly
    optional fast command-line appointment entry
    flexible ways to specify holidays and vacations
    extensive context help
    multiuser capability using an IP server program (with access lists),
    grouping of appointments into files, per-user, private, and others

    The action being taken when a warn or alarm time is reached is programmable; by default a window pops up. In addition, a program can be executed, or mail can be sent. Other methods of listing appointments (today, this week, next week, or a keyword search for regular expressions) are also available. Plan can be configured to display times in 12-hour or 24-hour formats, mmddyy and ddmmyy date formats, and can show either Monday or Sunday in the leftmost column. Four view modes are supported: month, year, week, day, and a 365-day overview. The day, week, and overview plot appointments as colored and labeled bars on a time chart.

        the date, time, and length (time and days) of the appointment,
        an optional text message to be printed,

Despite the duration is written but does not appears in all cases(?) and needs to be computed from differences, ex:

(1589846400 - 1589673600)/3600.0/24.0 = 2

that is not easy if there is a gap, this allocated time is not used further.

Then, using only plan as input requires a good formatting of the plan iD:

UserName-X-YY-ZZZ-xd-SampleEnv

for a normal experiment or:

UserName-TEST-XXXX-xd-SampleEnv
UserName-EASY-XXXX-xd-SampleEnv
...

For a test, internal or easy experiment.

In [27]:
from datetime import datetime, date
import sys
sys.path.append("/nethome/tofhr/ollivier/IN5/experiments_management/SoftTools/Gantt_Calendars/gantt-schedule/")
import src.libschedule as lg
from importlib import reload

In [25]:
reload(lg)

planfile = "/home/ollivier/Calendars/Allocated"  # <-- the definitive should be this one.
# planfile = "/home/tofhr/ollivier/Calendars/Propositions"# <-- while working with the planning. 

Exp, Date, Dur = lg.import_from_plan(planfile, PrintInfo=True)

Nb items in Exp:  662
Nb items in Date: 662
Nb items in Dur:  630


In [26]:

DF = lg. convert_to_dataframe(Date, Exp, Dur,startDate = '2026-06-01', endDate='2026-08-05' )
DF.head()

,Task,Start,Finish,Proposers,AllDays,SampleEnv,id_exp
0,7-02-253,2026-06-08,2026-06-11,Laven,3,CFCL,Laven-7-02-253-3d-CFCL
1,9-12-749,2026-06-11,2026-06-15,Pascariu,4,CF,Pascariu-9-12-749-4d-CF
2,7-03-275,2026-06-15,2026-06-18,Nilsen,3,CF,Nilsen-7-03-275-3d-CF
3,Magnet-Setup,2026-06-18,2026-06-19,IN5,1,MV10T,IN5-Magnet-Setup-1d-MV10T
4,7-01-675,2026-06-19,2026-06-22,Ueda,3,MV10T,Ueda-7-01-675-3d-MV10T


### Plot the Gantt from the dataFrame

The plot create a html with some interactive informations. Started fro inside `jupyter lab` or `jupyter notebook` it opens directly the html in the browser.

Ceate the Hollidays data frame.

In [28]:
reload(lg)
from datetime import date
ddate = datetime.strftime(date.today(), "%Y-%m-%d")

# Feries 2024
Feries = ["2024-03-29", "2024-04-01", "2024-05-01", "2024-05-08",
          "2024-05-09",  "2024-05-11", "2024-05-20"]

# Feries 2026
Feries = ["2026-04-03", "2026-04-06", "2026-05-01", "2026-05-08",
          "2026-05-14",  "2026-05-25", "2026-07-14", '2026-08-15']


Create the html plot and open it in a browser.

In [29]:
fig = lg.plot_gantt(DF, 
                    gantt_name=f'IN5_schedule_on_{ddate}', 
                    OffLine=True, 
                    Feries=Feries)